# 02 - Keypoint Visualization

This notebook loads a precomputed `.npy` keypoint file, visualizes the
keypoint sequence as an animation, and demonstrates the effects of
normalization and augmentation transforms.

**Prerequisites:**
- Run the preprocessing pipeline to generate `.npy` keypoint files in `data/processed/`.
- Or use the functions here to extract keypoints from a single video on the fly.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

%matplotlib inline

## 1. Load Keypoint Data

Each `.npy` file has shape `(T, 543, 3)` where T is the number of frames
and 543 = 33 pose + 21 left hand + 21 right hand + 468 face landmarks.
Each landmark has (x, y, z) coordinates normalized to [0, 1].

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"

# Find any available .npy file
npy_files = sorted(processed_dir.glob("*.npy"))

if len(npy_files) == 0:
    print("No .npy files found. Let's create a dummy example for demonstration.")
    # Create synthetic keypoint data for visualization
    T = 30
    keypoints = np.random.rand(T, 543, 3).astype(np.float32) * 0.3 + 0.35
    # Make hand keypoints cluster around a moving center
    for t in range(T):
        center = np.array([0.5 + 0.1 * np.sin(2 * np.pi * t / T), 0.4, 0])
        keypoints[t, 33:54] = center + np.random.randn(21, 3) * 0.02  # left hand
        keypoints[t, 54:75] = center + np.array([0.15, 0, 0]) + np.random.randn(21, 3) * 0.02  # right hand
    source = "synthetic"
else:
    keypoints = np.load(str(npy_files[0]))
    source = npy_files[0].name

print(f"Source: {source}")
print(f"Shape: {keypoints.shape}")
print(f"Value range: [{keypoints.min():.3f}, {keypoints.max():.3f}]")

## 2. Static Keypoint Plot

Visualize the pose and hand keypoints for a single frame.
We focus on the upper-body landmarks (pose + hands) since those are
most relevant for sign recognition.

In [ ]:
def plot_keypoints_frame(kps, frame_idx=0, title=""):
    """Plot pose and hand keypoints for a single frame."""
    frame = kps[frame_idx]  # (543, 3)
    
    pose = frame[:33, :2]       # (33, 2)
    left_hand = frame[33:54, :2]  # (21, 2)
    right_hand = frame[54:75, :2] # (21, 2)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot pose (skip face landmarks in pose: indices 0-10 are head)
    ax.scatter(pose[11:, 0], pose[11:, 1], c="blue", s=30, label="Pose", zorder=3)
    ax.scatter(pose[:11, 0], pose[:11, 1], c="lightblue", s=15, label="Head pose", zorder=3)
    
    # Pose connections (simplified)
    pose_connections = [
        (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),  # arms
        (11, 23), (12, 24), (23, 24),  # torso
    ]
    for a, b in pose_connections:
        ax.plot([pose[a, 0], pose[b, 0]], [pose[a, 1], pose[b, 1]], "b-", alpha=0.3)
    
    # Plot hands
    ax.scatter(left_hand[:, 0], left_hand[:, 1], c="red", s=20, label="Left Hand", zorder=3)
    ax.scatter(right_hand[:, 0], right_hand[:, 1], c="green", s=20, label="Right Hand", zorder=3)
    
    # Hand connections
    finger_connections = [(0, 1), (1, 2), (2, 3), (3, 4),   # thumb
                          (0, 5), (5, 6), (6, 7), (7, 8),   # index
                          (0, 9), (9, 10), (10, 11), (11, 12),  # middle
                          (0, 13), (13, 14), (14, 15), (15, 16),  # ring
                          (0, 17), (17, 18), (18, 19), (19, 20)]  # pinky
    for a, b in finger_connections:
        ax.plot([left_hand[a, 0], left_hand[b, 0]], [left_hand[a, 1], left_hand[b, 1]], "r-", alpha=0.3)
        ax.plot([right_hand[a, 0], right_hand[b, 0]], [right_hand[a, 1], right_hand[b, 1]], "g-", alpha=0.3)
    
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(1.1, -0.1)  # Invert Y axis (image coordinates)
    ax.set_aspect("equal")
    ax.legend(loc="upper right")
    ax.set_title(title or f"Frame {frame_idx}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig, ax

# Show the first, middle, and last frame
T = keypoints.shape[0]
for idx in [0, T // 2, T - 1]:
    plot_keypoints_frame(keypoints, frame_idx=idx, title=f"Frame {idx}/{T}")
    plt.show()

## 3. Animated Keypoint Sequence

Create a matplotlib animation showing the keypoints moving through time.
This gives an intuitive view of the signing motion.

In [ ]:
def animate_keypoints(kps, interval=100, max_frames=60):
    """Create a matplotlib animation of keypoint motion."""
    T = min(kps.shape[0], max_frames)
    kps = kps[:T]
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(1.1, -0.1)
    ax.set_aspect("equal")
    
    pose_scatter = ax.scatter([], [], c="blue", s=20)
    left_scatter = ax.scatter([], [], c="red", s=15)
    right_scatter = ax.scatter([], [], c="green", s=15)
    title_text = ax.set_title("")
    
    def init():
        pose_scatter.set_offsets(np.empty((0, 2)))
        left_scatter.set_offsets(np.empty((0, 2)))
        right_scatter.set_offsets(np.empty((0, 2)))
        return pose_scatter, left_scatter, right_scatter
    
    def update(frame_idx):
        frame = kps[frame_idx]
        pose_scatter.set_offsets(frame[:33, :2])
        left_scatter.set_offsets(frame[33:54, :2])
        right_scatter.set_offsets(frame[54:75, :2])
        title_text.set_text(f"Frame {frame_idx}/{T}")
        return pose_scatter, left_scatter, right_scatter
    
    anim = animation.FuncAnimation(
        fig, update, frames=T, init_func=init,
        interval=interval, blit=True
    )
    plt.close(fig)
    return anim

anim = animate_keypoints(keypoints)
HTML(anim.to_jshtml())

## 4. Effect of Normalization

Our normalization pipeline translates keypoints so the nose is at origin
and scales by shoulder width. This removes variation in signer position
and distance from camera.

In [ ]:
from src.data.preprocess import normalize_keypoints

kps_raw = keypoints.copy()
kps_norm = normalize_keypoints(kps_raw)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

mid = keypoints.shape[0] // 2

# Raw
frame_raw = kps_raw[mid]
axes[0].scatter(frame_raw[:33, 0], frame_raw[:33, 1], c="blue", s=15)
axes[0].scatter(frame_raw[33:54, 0], frame_raw[33:54, 1], c="red", s=10)
axes[0].scatter(frame_raw[54:75, 0], frame_raw[54:75, 1], c="green", s=10)
axes[0].set_title("Raw Keypoints")
axes[0].set_aspect("equal")
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)

# Normalized
frame_norm = kps_norm[mid]
axes[1].scatter(frame_norm[:33, 0], frame_norm[:33, 1], c="blue", s=15)
axes[1].scatter(frame_norm[33:54, 0], frame_norm[33:54, 1], c="red", s=10)
axes[1].scatter(frame_norm[54:75, 0], frame_norm[54:75, 1], c="green", s=10)
axes[1].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].axvline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_title("Normalized (nose-centered, shoulder-scaled)")
axes[1].set_aspect("equal")
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Effect of Augmentations

Visualize how each augmentation transform modifies the keypoint sequence.
These transforms help the model generalize by simulating variation in
signing speed, handedness, and noise.

In [ ]:
from src.data.augment import (
    TemporalCrop, TemporalFlip, TemporalSpeedPerturb,
    KeypointHorizontalFlip, KeypointNoise, KeypointScale,
    get_train_transforms,
)

kps_norm = normalize_keypoints(keypoints.copy())

augmentations = {
    "Original": kps_norm,
    "TemporalCrop(T=32)": TemporalCrop(T=32)(kps_norm.copy()),
    "TemporalFlip": TemporalFlip(p=1.0)(kps_norm.copy()),
    "SpeedPerturb(0.7-1.3)": TemporalSpeedPerturb(0.7, 1.3)(kps_norm.copy()),
    "HorizontalFlip": KeypointHorizontalFlip(p=1.0)(kps_norm.copy()),
    "Noise(sigma=0.02)": KeypointNoise(sigma=0.02)(kps_norm.copy()),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, (name, kps) in zip(axes, augmentations.items()):
    mid = kps.shape[0] // 2
    frame = kps[mid]
    ax.scatter(frame[:33, 0], frame[:33, 1], c="blue", s=10)
    ax.scatter(frame[33:54, 0], frame[33:54, 1], c="red", s=8)
    ax.scatter(frame[54:75, 0], frame[54:75, 1], c="green", s=8)
    ax.set_title(f"{name} (T={kps.shape[0]})")
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show temporal effect: plot the x-coordinate of the right wrist over time
fig, ax = plt.subplots(figsize=(12, 4))
for name, kps in augmentations.items():
    wrist_x = kps[:, 16, 0]  # Pose landmark 16 = right wrist
    ax.plot(wrist_x, label=name, alpha=0.7)
ax.set_xlabel("Frame")
ax.set_ylabel("Right Wrist X")
ax.set_title("Temporal Effect of Augmentations on Right Wrist X-Coordinate")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()